# 04 · Política de crédito na Base C (exercício guiado)

**Objetivo:** transformar o score em decisão (a quem conceder, a que taxa, em que prazo e com quanta entrada) e
levar uma proposta para comparar com a do colega responsável pela política. **Não é a submissão oficial.**

**Limites do conselho:** aprovação ≥ 35% das 5.000 propostas · inadimplência ≤ 8% dos contratos ·
volume contratado ≥ R$ 40 mi · taxa ≤ 3,5% a.m. **Meta:** ROI anual acima de 15%.

**ROI anual** = [(juros recebidos − perda) ÷ volume financiado] ÷ prazo médio em anos. Quem quebra paga juros
só até o mês do calote; proposta aprovada que o cliente recusa não entra na conta.

**O problema:** a Base C reage à oferta. Taxa alta afasta o cliente e atrai quem não tem alternativa
(seleção adversa); entrada e prazo curto também derrubam o aceite. O enunciado dá a direção desses efeitos, mas
não a intensidade. Por isso simulamos **três cenários** e exigimos que a política cumpra os limites mesmo no severo.

In [1]:
import json

import joblib
import pandas as pd

from autocred import dados, politica

pd.set_option("display.width", 220)
SAIDAS = dados.PASTA_SAIDAS
pacote = joblib.load(SAIDAS / "modelo_pd.joblib")
modelo, cortes = pacote["modelo_portatil"], pacote["cortes"]
base = politica.preparar_base(modelo, dados.carregar_base("C"), cortes)


def resumo(nome, regras, margem=1.0):
    p = politica.painel(modelo, base, cortes, regras, margem)
    return {
        "alternativa": nome,
        "aprov.": f"{p.aprovacao.iloc[0]:.1%}",
        "vol. base (R$ mi)": round(p.loc["base", "volume"] / 1e6, 1),
        "vol. severo (R$ mi)": round(p.loc["severo", "volume"] / 1e6, 1),
        "inad. base": f"{p.loc['base', 'inadimplencia']:.1%}",
        "inad. severo": f"{p.loc['severo', 'inadimplencia']:.1%}",
        "ROI base": f"{p.loc['base', 'roi']:.1%}",
        "ROI severo": f"{p.loc['severo', 'roi']:.1%}",
        "limites ok (brando/base/severo)": "/".join("sim" if v else "NÃO" for v in p.ok_limites),
    }


def comparar(alternativas, margem=1.0):
    display(pd.DataFrame([resumo(n, r, margem) for n, r in alternativas]))

## 1. Premissas de reação do cliente

A taxa de referência é a média antiga, 1,589% a.m. Os efeitos valem por +1 ponto de taxa ao mês acima dela,
por +10 pontos de entrada exigida além da desejada e por 12 meses de prazo cortados.

In [2]:
display(pd.DataFrame([c.__dict__ for c in politica.CENARIOS]).set_index("nome"))
efeito = []
for taxa in [0.0159, 0.020, 0.025, 0.030, 0.035]:
    linha = {"taxa a.m.": f"{taxa:.2%}"}
    for c in politica.CENARIOS:
        aceite = politica.aceite(c, taxa, 0, 0) / c.aceite_base
        mult_pd = (1 + c.selecao_adversa) ** max(0.0, (taxa - politica.TAXA_REFERENCIA_AM) * 100)
        linha[c.nome] = f"aceite {aceite:.0%} · PD ×{mult_pd:.2f}"
    efeito.append(linha)
display(pd.DataFrame(efeito))

,aceite_base,fator_taxa,queda_entrada,queda_prazo,selecao_adversa
nome,,,,,
brando,0.90,0.86,0.05,0.05,0.10
base,0.85,0.74,0.10,0.10,0.22
severo,0.80,0.61,0.15,0.15,0.42


,taxa a.m.,brando,base,severo
0,1.59%,aceite 100% · PD ×1.00,aceite 100% · PD ×1.00,aceite 100% · PD ×1.00
1,2.00%,aceite 94% · PD ×1.04,aceite 88% · PD ×1.09,aceite 82% · PD ×1.16
2,2.50%,aceite 87% · PD ×1.09,aceite 76% · PD ×1.20,aceite 64% · PD ×1.38
3,3.00%,aceite 81% · PD ×1.14,aceite 65% · PD ×1.32,aceite 50% · PD ×1.64
4,3.50%,aceite 75% · PD ×1.20,aceite 56% · PD ×1.46,aceite 39% · PD ×1.95


**Três políticas ingênuas, para calibrar a intuição:**

In [3]:
comparar([
    ("A) aprovar todos a 1,59%", politica.montar_regras(1, 0.0159)),
    ("B) scores 5–10 a 1,59%", politica.montar_regras(5, 0.0159)),
    ("C) scores 5–10 a 3,5%", politica.montar_regras(5, 0.035)),
])

,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,"A) aprovar todos a 1,59%",100.0%,142.0,133.6,14.5%,14.5%,7.2%,7.2%,NÃO/NÃO/NÃO
1,"B) scores 5–10 a 1,59%",36.9%,59.1,55.6,4.3%,4.3%,9.8%,9.8%,sim/sim/sim
2,"C) scores 5–10 a 3,5%",36.9%,33.3,21.6,6.9%,9.2%,24.2%,23.2%,sim/NÃO/NÃO


- **A** repete o problema do caso: sem filtro, a inadimplência passa de 14% e o ROI fica em ~7%.
- **B**: filtrar resolve o risco, mas o preço antigo não entrega os 15%.
- **C**: cobrar o teto dá ROI alto, mas afasta os clientes (volume abaixo de R$ 40 mi) e, no severo, a seleção
  adversa estoura a inadimplência. É a "política degenerada" do enunciado.

**Decisão 1:** manter as premissas como estão. O severo já é duro: a 3,5% a.m., perde 61% dos clientes e quase dobra a PD.

## 2. Corte: quais scores aprovar (taxa provisória de 2,5% a.m.)

In [4]:
comparar([(f"scores {s}–10", politica.montar_regras(s, 0.025)) for s in [7, 6, 5, 4, 3, 2]])

,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,scores 7–10,23.6%,29.0,22.9,4.4%,5.0%,16.8%,16.6%,NÃO/NÃO/NÃO
1,scores 6–10,30.8%,38.0,30.0,4.9%,5.6%,16.7%,16.4%,NÃO/NÃO/NÃO
2,scores 5–10,36.9%,44.9,35.5,5.3%,6.1%,16.5%,16.2%,sim/sim/NÃO
3,scores 4–10,44.8%,53.8,42.4,6.0%,6.9%,16.2%,15.9%,sim/sim/sim
4,scores 3–10,53.5%,63.6,50.2,6.8%,7.9%,15.9%,15.6%,sim/sim/sim
5,scores 2–10,64.4%,75.4,59.5,8.3%,9.5%,15.4%,15.0%,sim/NÃO/NÃO


**Decisão 2: aprovar os scores 4 a 10.** É o corte que cumpre os limites nos três cenários com folga dos dois lados:
o 5–10 fura o volume no severo, e o 3–10 deixa a inadimplência do severo colada nos 8%.

## 3. Taxa por faixa

Primeiro, a economia de cada faixa. Com a definição de ROI do enunciado, a perda (que acontece uma vez) é diluída
pelos ~3,5 anos de prazo médio, então **o nível da taxa pesa muito mais que a faixa**.

In [5]:
for cenario in politica.CENARIOS[1:]:
    tabela = {}
    for taxa in [0.020, 0.022, 0.024, 0.026, 0.028, 0.030, 0.035]:
        _, det = politica.simular(modelo, base, cortes, politica.montar_regras(4, taxa), cenario)
        w = det["aceite"]
        g = det.assign(
            wj=w * det["juros_esperados"], wp=w * det["perda_esperada"], wv=w * det["valor_financiado"],
            wz=w * det["prazo_meses"], w=w,
        ).groupby("score_1a10")[["wj", "wp", "wv", "wz", "w"]].sum()
        tabela[f"{taxa:.1%}"] = ((g["wj"] - g["wp"]) / g["wv"] / (g["wz"] / g["w"] / 12)).map("{:.0%}".format)
    print(f"ROI por faixa, cenário {cenario.nome}")
    display(pd.DataFrame(tabela).sort_index(ascending=False))

ROI por faixa, cenário base


,2.0%,2.2%,2.4%,2.6%,2.8%,3.0%,3.5%
score_1a10,,,,,,,
10,13%,15%,16%,18%,20%,21%,25%
9,13%,15%,16%,18%,19%,21%,25%
8,13%,14%,16%,17%,19%,21%,25%
7,13%,14%,16%,17%,19%,20%,24%
6,12%,14%,15%,17%,18%,20%,24%
5,12%,13%,15%,16%,18%,19%,23%
4,12%,13%,14%,16%,17%,18%,22%


ROI por faixa, cenário severo


,2.0%,2.2%,2.4%,2.6%,2.8%,3.0%,3.5%
score_1a10,,,,,,,
10,13%,15%,16%,18%,19%,21%,25%
9,13%,15%,16%,18%,19%,21%,24%
8,13%,14%,16%,17%,19%,20%,24%
7,13%,14%,15%,17%,18%,20%,23%
6,12%,14%,15%,16%,18%,19%,22%
5,12%,13%,14%,16%,17%,18%,21%
4,11%,13%,14%,15%,16%,17%,20%


In [6]:
ESCADA = {1: 0.035, 2: 0.035, 3: 0.035, 4: 0.029, 5: 0.028, 6: 0.026, 7: 0.025, 8: 0.024, 9: 0.023, 10: 0.023}
comparar([
    ("plana 2,4%", politica.montar_regras(4, 0.024)),
    ("plana 2,5%", politica.montar_regras(4, 0.025)),
    ("plana 2,6%", politica.montar_regras(4, 0.026)),
    ("plana 2,7%", politica.montar_regras(4, 0.027)),
    ("escada por risco 2,3% → 2,9%", politica.montar_regras(4, ESCADA)),
])

,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,"plana 2,4%",44.8%,55.4,44.6,5.8%,6.6%,15.5%,15.2%,sim/sim/sim
1,"plana 2,5%",44.8%,53.8,42.4,6.0%,6.9%,16.2%,15.9%,sim/sim/sim
2,"plana 2,6%",44.8%,52.2,40.4,6.1%,7.2%,17.0%,16.6%,sim/sim/sim
3,"plana 2,7%",44.8%,50.6,38.5,6.3%,7.5%,17.7%,17.3%,sim/sim/NÃO
4,"escada por risco 2,3% → 2,9%",44.8%,53.1,41.6,6.1%,7.0%,16.5%,16.1%,sim/sim/sim


Cada +0,1 ponto de taxa rende ~+0,8 ponto de ROI e custa ~R$ 2 mi de volume no severo. A 2,7% o volume já fura o piso.

**Decisão 3: escada por risco.** Scores 10–9 pagam 2,3%; 8: 2,4%; 7: 2,5%; 6: 2,6%; 5: 2,8%; 4: 2,9% a.m. Rende o
mesmo que a plana de 2,5% e conta a história certa para a defesa: a taxa sobe conforme a perda esperada da faixa.

## 4. Prazo máximo

In [7]:
def por_score(padrao, especificos):
    return {s: especificos.get(s, padrao) for s in range(1, 11)}


comparar([
    ("sem limite (60 m)", politica.montar_regras(4, ESCADA)),
    ("48 m para todos", politica.montar_regras(4, ESCADA, prazo_max=48)),
    ("48 m nos scores 4–6", politica.montar_regras(4, ESCADA, prazo_max=por_score(60, {4: 48, 5: 48, 6: 48}))),
    ("36 m nos 4–5, 48 m nos 6–7", politica.montar_regras(4, ESCADA, prazo_max=por_score(60, {4: 36, 5: 36, 6: 48, 7: 48}))),
])

,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,sem limite (60 m),44.8%,53.1,41.6,6.1%,7.0%,16.5%,16.1%,sim/sim/sim
1,48 m para todos,44.8%,51.9,40.2,6.1%,7.0%,16.2%,15.8%,sim/sim/sim
2,48 m nos scores 4–6,44.8%,52.6,41.1,6.1%,7.0%,16.3%,16.0%,sim/sim/sim
3,"36 m nos 4–5, 48 m nos 6–7",44.8%,51.7,40.1,6.0%,6.8%,16.1%,15.8%,sim/sim/sim


Encurtar o prazo **não reduziu a inadimplência**. A parcela sobe e o comprometimento de renda (4ª variável mais
importante do modelo) compensa o prazo menor. Só custou aceite, volume e ROI, porque a perda passa a ser
diluída em menos anos.

**Decisão 4: sem limite de prazo (60 meses para todos).**

## 5. Entrada mínima

In [8]:
comparar([
    ("sem mínimo", politica.montar_regras(4, ESCADA)),
    ("10% para todos", politica.montar_regras(4, ESCADA, entrada_min=0.10)),
    ("20% para todos", politica.montar_regras(4, ESCADA, entrada_min=0.20)),
    ("20% nos scores 4–5", politica.montar_regras(4, ESCADA, entrada_min=por_score(0.0, {4: 0.20, 5: 0.20}))),
])

,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,sem mínimo,44.8%,53.1,41.6,6.1%,7.0%,16.5%,16.1%,sim/sim/sim
1,10% para todos,44.8%,52.6,41.2,5.9%,6.8%,16.5%,16.2%,sim/sim/sim
2,20% para todos,44.8%,49.2,37.9,5.9%,6.7%,16.6%,16.2%,sim/sim/NÃO
3,20% nos scores 4–5,44.8%,51.6,40.3,5.9%,6.8%,16.5%,16.1%,sim/sim/sim


No nosso modelo o LTV quase não pesa na PD, então o ganho vem pela LGD e é pequeno. Mas o enunciado diz que, no
simulador, LTV menor reduz PD **e** LGD, então o efeito real pode ser maior. Com 10%, só 12% dos aprovados são
afetados, e quem pede LTV acima de 95% volta ao patamar que a AutoCred já conhecia.

**Decisão 5: entrada mínima de 10% para todas as faixas.** É um seguro barato: −0,2 ponto de inadimplência e só
−R$ 0,4 mi de volume no severo.

## 6. Propostas fora do perfil histórico

In [9]:
def regras_com(aceita):
    return politica.montar_regras(4, ESCADA, entrada_min=0.10, aceita_fora_perfil=aceita)


aprovados = base[base["score_1a10"] >= 4]
print(f"Aprovados fora do perfil: {aprovados['fora_perfil'].sum()} de {len(aprovados)} ({aprovados['fora_perfil'].mean():.1%})")
negar_4_6 = por_score(True, {4: False, 5: False, 6: False})
comparar([
    ("aceitar (PD do modelo)", regras_com(True)),
    ("negar fora do perfil nos 4–6", regras_com(negar_4_6)),
    ("negar fora do perfil em todas", regras_com(False)),
])
print("Estresse: a mesma regra de aceitar, avaliada com a PD dos fora do perfil dobrada")
comparar([("aceitar, estresse PD ×2", regras_com(True))], margem=2.0)

Aprovados fora do perfil: 198 de 2241 (8.8%)


,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,aceitar (PD do modelo),44.8%,52.6,41.2,5.9%,6.8%,16.5%,16.2%,sim/sim/sim
1,negar fora do perfil nos 4–6,41.7%,49.7,39.1,5.8%,6.7%,16.4%,16.1%,sim/sim/NÃO
2,negar fora do perfil em todas,40.9%,48.8,38.3,5.8%,6.7%,16.4%,16.1%,sim/sim/NÃO


Estresse: a mesma regra de aceitar, avaliada com a PD dos fora do perfil dobrada


,alternativa,aprov.,vol. base (R$ mi),vol. severo (R$ mi),inad. base,inad. severo,ROI base,ROI severo,limites ok (brando/base/severo)
0,"aceitar, estresse PD ×2",44.8%,52.6,41.2,6.5%,7.5%,16.4%,16.0%,sim/sim/sim


Mesmo dobrando a PD de quem o modelo nunca viu, a política cumpre os limites. Negar esses casos custa aprovação e
fura o volume no severo, sem melhorar o ROI.

**Decisão 6: aceitar pela regra da faixa e avaliar a política final com a PD dos fora do perfil dobrada.**

## 7. Checagem final e apetite de risco

In [10]:
FINAL = politica.montar_regras(4, ESCADA, prazo_max=60, entrada_min=0.10, aceita_fora_perfil=True)
DELTA_AGRESSIVA = 0.003
AGRESSIVA = politica.montar_regras(
    4, {s: min(0.035, t + DELTA_AGRESSIVA) for s, t in ESCADA.items()}, prazo_max=60, entrada_min=0.10, aceita_fora_perfil=True
)
MARGEM = 2.0
paineis = {}
for chave, nome, regras in [
    ("final", "Política final (robusta no severo)", FINAL),
    ("agressiva", "Variante agressiva (+0,3 ponto)", AGRESSIVA),
]:
    p = politica.painel(modelo, base, cortes, regras, MARGEM)
    paineis[chave] = p
    print(nome)
    display(pd.DataFrame({
        "aprovação": p["aprovacao"].map("{:.1%}".format),
        "contratos": p["contratos"].round().astype(int),
        "volume (R$ mi)": (p["volume"] / 1e6).round(1),
        "inadimplência": p["inadimplencia"].map("{:.1%}".format),
        "prazo médio (anos)": p["prazo_medio_anos"].round(2),
        "ROI a.a.": p["roi"].map("{:.1%}".format),
        "limites ok": p["ok_limites"],
    }))

Política final (robusta no severo)


,aprovação,contratos,volume (R$ mi),inadimplência,prazo médio (anos),ROI a.a.,limites ok
cenario,,,,,,,
brando,44.8%,1740,64.4,5.9%,3.51,16.6%,True
base,44.8%,1420,52.6,6.5%,3.51,16.4%,True
severo,44.8%,1110,41.2,7.5%,3.51,16.0%,True


Variante agressiva (+0,3 ponto)


,aprovação,contratos,volume (R$ mi),inadimplência,prazo médio (anos),ROI a.a.,limites ok
cenario,,,,,,,
brando,44.8%,1663,61.5,6.2%,3.51,18.9%,True
base,44.8%,1297,48.1,7.0%,3.51,18.6%,True
severo,44.8%,957,35.5,8.4%,3.51,18.0%,False


A variante agressiva sobe o ROI para ~18,6% no base, mas, se o cliente for tão sensível quanto no severo, fura a
inadimplência e o volume. Cada limite furado corta a nota da política pela metade.

**Decisão 7: política robusta.** ROI de 16,0% a 16,6% e todos os limites cumpridos nos três cenários, com o estresse.

### A tabela de regras

In [11]:
_, detalhe = politica.simular(modelo, base, cortes, FINAL, politica.CENARIOS[1], MARGEM)
w = detalhe["aceite"]
g = detalhe.assign(
    wj=w * detalhe["juros_esperados"], wp=w * detalhe["perda_esperada"], wv=w * detalhe["valor_financiado"],
    wz=w * detalhe["prazo_meses"], w=w, wpd=w * detalhe["pd_simulada"],
).groupby("score_1a10")[["wj", "wp", "wv", "wz", "w", "wpd"]].sum()
tabela_final = FINAL.copy()
tabela_final["propostas_C"] = base["score_1a10"].value_counts().reindex(tabela_final.index, fill_value=0)
tabela_final["aceite_base"] = (g["w"] / tabela_final["propostas_C"]).reindex(tabela_final.index)
tabela_final["pd_simulada_base"] = (g["wpd"] / g["w"]).reindex(tabela_final.index)
tabela_final["roi_base"] = ((g["wj"] - g["wp"]) / g["wv"] / (g["wz"] / g["w"] / 12)).reindex(tabela_final.index)
display(tabela_final.sort_index(ascending=False))

,aprovar,taxa_am,prazo_max,entrada_min,aceita_fora_perfil,propostas_C,aceite_base,pd_simulada_base,roi_base
score_1a10,,,,,,,,,
10,True,0.023,60,0.1,True,307,0.685672,0.029932,0.156032
9,True,0.023,60,0.1,True,301,0.685499,0.039890,0.154005
8,True,0.024,60,0.1,True,274,0.664281,0.048381,0.158881
7,True,0.025,60,0.1,True,299,0.644406,0.058809,0.163847
6,True,0.026,60,0.1,True,360,0.624013,0.069633,0.167219
5,True,0.028,60,0.1,True,302,0.586756,0.089176,0.174832
4,True,0.029,60,0.1,True,398,0.568483,0.114585,0.172339
3,False,0.035,60,0.1,True,434,NaN,NaN,NaN
2,False,0.035,60,0.1,True,546,NaN,NaN,NaN


## 8. Saídas

In [12]:
tabela_final.to_csv(SAIDAS / "tabela_politica.csv", encoding="utf-8")
submissao = politica.arquivo_submissao(base, FINAL)
politica.validar_coerencia(submissao, base, FINAL, cortes)
submissao.to_csv(SAIDAS / "politica_base_C.csv", index=False)
assert len(submissao) == 5000
print(submissao["decisao"].value_counts().to_dict())
display(submissao.head())

# Números para o resumo de segunda (scripts/gerar_resumo.py lê este arquivo).
resultados_politica = {
    "margem_fora_perfil": MARGEM,
    "delta_agressiva": DELTA_AGRESSIVA,
    "cenarios": [c.__dict__ for c in politica.CENARIOS],
    "final": paineis["final"].reset_index().to_dict("records"),
    "agressiva": paineis["agressiva"].reset_index().to_dict("records"),
    "regras": tabela_final.reset_index().to_dict("records"),
    "decisoes": submissao["decisao"].value_counts().to_dict(),
}
with open(SAIDAS / "resultados_politica.json", "w", encoding="utf-8") as arquivo:
    json.dump(resultados_politica, arquivo, ensure_ascii=False, indent=2)

{'NEGAR': 2759, 'APROVAR': 2241}


,id_proposta,pd,score_1a10,decisao,taxa_am,prazo_meses,pct_entrada_minima
0,P000001,0.225628,1,NEGAR,NaN,<NA>,NaN
1,P000002,0.053787,6,APROVAR,0.026,48,0.1
2,P000003,0.015876,10,APROVAR,0.023,24,0.1
3,P000004,0.108449,2,NEGAR,NaN,<NA>,NaN
4,P000005,0.399601,1,NEGAR,NaN,<NA>,NaN
